# Stage 01 — Data Collection

Fetches ~20 years of daily S&P 500 (`^GSPC`) OHLCV data, cleans it, and persists **both** the raw and cleaned datasets as Parquet to the project Shared Drive `data/` folder (distinguished by the `_raw` / `_clean` filename suffix).

Pipeline: `fetch_ohlcv` → `save_raw` → `clean_ohlcv` → `save_clean` → verify.

## Environment setup (Google Colab)

Mounts the Shared Drive, clones the repo using the `GITHUB_TOKEN` Colab secret, puts the repo root on `sys.path`, and installs dependencies.

In [ ]:
import os
import sys

from google.colab import drive, userdata

# 1. Mount Google Drive for datasets & model weights
drive.mount('/content/drive')

# 2. Clone the (private) repo using the GITHUB_TOKEN Colab secret
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER = 'jesseingraham'
REPO_NAME = 'comp-653-stock-prediction-i'
REPO_PATH = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_PATH):
    !git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git

# 3. Work from the repo root and put it on the import path
os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

# 4. Install dependencies
!pip install -r requirements.txt

## Imports & configuration

Core logic lives in `src/`; this notebook only orchestrates and visualizes.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

from config import DRIVE_DATA_PATH, END_DATE, START_DATE, TICKER
from src.data.fetcher import RAW_FILENAME, fetch_ohlcv, save_raw
from src.data.cleaner import CLEAN_FILENAME, clean_ohlcv, save_clean

print(f'Ticker: {TICKER} | {START_DATE} -> {END_DATE}')
print(f'Data folder: {DRIVE_DATA_PATH}')

## 1. Fetch raw OHLCV and persist

Downloads daily bars from Yahoo Finance and writes `sp500_daily_raw.parquet` to the Drive `data/` folder.

In [ ]:
raw = fetch_ohlcv()
raw_path = save_raw(raw)
print(f'Raw: {raw.shape[0]:,} rows x {raw.shape[1]} cols -> {raw_path}')
raw.head()

## 2. Clean and persist

Validates and repairs the raw frame (see `src/data/cleaner.py`) and writes `sp500_daily_clean.parquet` alongside the raw file.

In [ ]:
clean = clean_ohlcv(raw)
clean_path = save_clean(clean)
print(f'Clean: {clean.shape[0]:,} rows x {clean.shape[1]} cols -> {clean_path}')
clean.head()

## 3. Verify persisted files

Reload both Parquet files from the Drive to confirm they round-trip, then summarize the cleaned series.

In [ ]:
raw_reloaded = pd.read_parquet(os.path.join(DRIVE_DATA_PATH, RAW_FILENAME))
clean_reloaded = pd.read_parquet(
    os.path.join(DRIVE_DATA_PATH, CLEAN_FILENAME)
)

print('Raw reloaded:  ', raw_reloaded.shape)
print('Clean reloaded:', clean_reloaded.shape)
clean_reloaded.describe()

In [ ]:
features = list(clean_reloaded.columns)
fig, axes = plt.subplots(
    len(features), 1, figsize=(12, 2.4 * len(features)), sharex=True
)
for ax, col in zip(axes, features):
    ax.plot(clean_reloaded.index, clean_reloaded[col], linewidth=0.8)
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)
axes[0].set_title('S&P 500 (^GSPC) Daily — Cleaned Features')
axes[-1].set_xlabel('Date')
fig.tight_layout()
plt.show()

## Summary

The raw and cleaned S&P 500 daily OHLCV datasets are now persisted to the Shared Drive `data/` folder. **Next:** Stage 02 — feature engineering (chart-pattern detection and look-back windowing).